# Lab 6 - Classification

- Họ và tên:

- MSSV: 

## I. Hướng dẫn

### Khởi tạo Spark

In [1]:
import findspark
findspark.init()

import pyspark
findspark.find()

from pyspark.sql import SparkSession
from pyspark.sql.functions import count

spark = (SparkSession
         .builder
         .appName("Classification")
         .getOrCreate())

### Đọc và load tập dữ liệu Iris

In [2]:
irisDF = (spark.read
          .option("HEADER", True)
          .option("inferSchema", True)
          .csv("./data/iris.csv")
         )

irisDF.show(5)

+------------+-----------+------------+-----------+-----------+
|sepal_length|sepal_width|petal_length|petal_width|      class|
+------------+-----------+------------+-----------+-----------+
|         5.1|        3.5|         1.4|        0.2|Iris-setosa|
|         4.9|        3.0|         1.4|        0.2|Iris-setosa|
|         4.7|        3.2|         1.3|        0.2|Iris-setosa|
|         4.6|        3.1|         1.5|        0.2|Iris-setosa|
|         5.0|        3.6|         1.4|        0.2|Iris-setosa|
+------------+-----------+------------+-----------+-----------+
only showing top 5 rows


### Chuyển cột `class` (kiểu string) thành `label` (kiểu double)

In [3]:
from pyspark.ml.feature import StringIndexer

class_indexer = StringIndexer(inputCol = 'class', outputCol = 'label')

irisDFindexed = class_indexer.fit(irisDF).transform(irisDF)

irisDFindexed.show(5)

+------------+-----------+------------+-----------+-----------+-----+
|sepal_length|sepal_width|petal_length|petal_width|      class|label|
+------------+-----------+------------+-----------+-----------+-----+
|         5.1|        3.5|         1.4|        0.2|Iris-setosa|  0.0|
|         4.9|        3.0|         1.4|        0.2|Iris-setosa|  0.0|
|         4.7|        3.2|         1.3|        0.2|Iris-setosa|  0.0|
|         4.6|        3.1|         1.5|        0.2|Iris-setosa|  0.0|
|         5.0|        3.6|         1.4|        0.2|Iris-setosa|  0.0|
+------------+-----------+------------+-----------+-----------+-----+
only showing top 5 rows


### Tập dữ liệu Iris

`sepal_length`: chiều dài đài hoa (cm)

`sepal_width`: chiều rộng đài hoa (cm)

`petal_length`: chiều dài cánh hoa (cm)

`petal_width`: chiều rộng cánh hoa (cm)

`class/label`: loại hoa

![Iris dataset](./image/iris.png)

### Chia dữ liệu thành train/test set

In [4]:
(trainDF, testDF) = irisDFindexed.randomSplit([.8, .2], seed = 1)

### Xem các loại biến trong tập dữ liệu

In [5]:
irisDFindexed.dtypes

[('sepal_length', 'double'),
 ('sepal_width', 'double'),
 ('petal_length', 'double'),
 ('petal_width', 'double'),
 ('class', 'string'),
 ('label', 'double')]

### Biến đổi train data và test data theo định dạng của Spark

In [6]:
from pyspark.ml.feature import VectorAssembler

assembler = VectorAssembler(inputCols = ['sepal_length','sepal_width','petal_length','petal_width'],
                            outputCol = 'features')
assembler_train = assembler.transform(trainDF)

X_train = assembler_train.select('features', 'label')
X_train.show(5)

+-----------------+-----+
|         features|label|
+-----------------+-----+
|[4.3,3.0,1.1,0.1]|  0.0|
|[4.4,2.9,1.4,0.2]|  0.0|
|[4.4,3.0,1.3,0.2]|  0.0|
|[4.4,3.2,1.3,0.2]|  0.0|
|[4.6,3.1,1.5,0.2]|  0.0|
+-----------------+-----+
only showing top 5 rows


## Sử dụng Logistic Regression

### 1.1 Tạo mô hình Logistic Regression

Tạo một một hình Logistic Regression và huấn luyện mô hình trên `X_train` với `labelCol` là `'label'` và `featuresCol` là `'features'`

In [7]:
from pyspark.ml.classification import LogisticRegression

logit = LogisticRegression(featuresCol = "features", labelCol = "label")

logitModel = logit.fit(X_train)

### 1.2. Áp dụng mô hình trên test data

Áp dụng biến đổi cho tập test tương tự như trên tập train. In ra vài dòng sau khi biến đổi để xem kết quả.

In [8]:
assembler_test = assembler.transform(testDF)
X_test = assembler_test.select('features', 'label')
X_test.show(5)

+-----------------+-----+
|         features|label|
+-----------------+-----+
|[4.5,2.3,1.3,0.3]|  0.0|
|[4.8,3.1,1.6,0.2]|  0.0|
|[4.8,3.4,1.6,0.2]|  0.0|
|[4.8,3.4,1.9,0.2]|  0.0|
|[4.9,2.5,4.5,1.7]|  2.0|
+-----------------+-----+
only showing top 5 rows


Dự đoán trên test data

In [9]:
predictions = logitModel.transform(X_test)
predictions.select("prediction", "label").show(5)

+----------+-----+
|prediction|label|
+----------+-----+
|       0.0|  0.0|
|       0.0|  0.0|
|       0.0|  0.0|
|       0.0|  0.0|
|       1.0|  2.0|
+----------+-----+
only showing top 5 rows


### 1.3. Đánh giá mô hình

Tính giá trị `Accuracy` của mô hình trên tập test

In [10]:
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

evaluator = MulticlassClassificationEvaluator()

accuracy = evaluator.evaluate(predictions, {evaluator.metricName: "accuracy"})
print("Accuracy = %g" % accuracy)
print("Test Error = %g" % (1.0 - accuracy))

Accuracy = 0.961538
Test Error = 0.0384615


### 1.4. Tạo ML pipeline và đánh giá dùng phương pháp cross validation

![cross-validation-model-selection](./image/cross_val.png)

In [11]:
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.feature import StringIndexer
from pyspark.ml import Pipeline
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder
from pyspark.ml.evaluation import MulticlassClassificationEvaluator

import pprint

pp = pprint.PrettyPrinter(indent = 4)

# Create a LogisticRegression instance. This instance is an Estimator.
logit = LogisticRegression(featuresCol = "features", labelCol = "label")

# Define indexer
indexer = StringIndexer(inputCol = 'class', 
                        outputCol = 'label')

# Define assembler
assembler = VectorAssembler(inputCols = ['sepal_length','sepal_width','petal_length','petal_width'],
                            outputCol = 'features')

# Configure an ML pipeline, which consists of two stages: indexer, assembler, and logit.
pipeline = Pipeline(stages = [indexer, assembler, logit])

# Specify evaluator
evaluator = MulticlassClassificationEvaluator(
    labelCol = "label", 
    predictionCol = "prediction",
    metricName = "accuracy"
)

# Specify parameters
paramGrid = (ParamGridBuilder()
            .addGrid(logit.regParam , [0.01, 0.1, 1])
            .build())

# Train/test split
(trainDF, testDF) = irisDF.randomSplit([.8, .2], seed = 1)

# Setup CrossValidator 
# A CrossValidator requires an Estimator, a set of Estimator ParamMaps, and an Evaluator.
cv = CrossValidator(estimator = logit, 
                    evaluator = evaluator, 
                    estimatorParamMaps = paramGrid, 
                    numFolds = 3, 
                    parallelism = 2, 
                    seed = 1)

# Run cross-validation on training data, and choose the best set of parameters
logitModel = pipeline.fit(trainDF)

# Make predictions on test data. logitModel uses the best model found (regParam = 0.1)
prediction = logitModel.transform(testDF)
result = prediction.select("features", "label", "prediction").collect()

# Print some predictions
for row in result[0:5]:
    pp.pprint("features=%s, label=%s -> prediction=%s" % 
              (row.features, row.label, row.prediction))

accuracy = evaluator.evaluate(predictions)

print("Test Error = %g" % (1.0 - accuracy))

'features=[4.5,2.3,1.3,0.3], label=2.0 -> prediction=2.0'
'features=[4.8,3.1,1.6,0.2], label=2.0 -> prediction=2.0'
'features=[4.8,3.4,1.6,0.2], label=2.0 -> prediction=2.0'
'features=[4.8,3.4,1.9,0.2], label=2.0 -> prediction=2.0'
'features=[4.9,2.5,4.5,1.7], label=1.0 -> prediction=0.0'
Test Error = 0.0384615


# II. Áp dụng

## Câu 1 - Áp dụng `LogisticRegression` với tập dữ liệu `Auto`

Câu hỏi này sử dụng Logistic Regression trên tập dữ liệu `Auto` để dự đoán một xe cho trước có `mpg` là `high` hay `low`.

**Auto Data Set Description**

A data frame with 392 observations on the following 9 variables.

- `mpg`: miles per gallon

- `cylinders`: Number of cylinders between 4 and 8

- `displacement`: Engine displacement (cu. inches)

- `horsepower`: Engine horsepower

- `weight`: Vehicle weight (lbs.)

- `acceleration`: Time to accelerate from 0 to 60 mph (sec.)

- `year`: Model year (modulo 100)

- `origin`: Origin of car (1. American, 2. European, 3. Japanese)

- `name`: Vehicle name

**1.1.** Tạo một binary variable nhận giá trị 1 (`high`) với các xe có `mpg` lớn hơn median mpg, và nhận giá trị 0 (`low`) cho các xe còn lại.

In [24]:
# Viết code của bạn ở đây
from pyspark.sql import functions as F

# 1) Load Auto dataset
auto_raw = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("./data/Auto.csv")
)

# 2) Clean horsepower (hay bị "?" -> null) rồi cast double
auto = (auto_raw
    .withColumn(
        "horsepower",
        F.when(F.col("horsepower").cast("string") == "?", F.lit(None))
         .otherwise(F.col("horsepower").cast("double"))
    )
    .dropna(subset=["mpg", "horsepower", "displacement", "weight", "acceleration", "year", "cylinders", "origin"])
)

# 3) Median mpg
median_mpg = auto.approxQuantile("mpg", [0.5], 0.0)[0]
print("Median mpg =", median_mpg)

# 4) Create binary label: 1 (high) if mpg > median else 0 (low)
auto = auto.withColumn(
    "label",
    F.when(F.col("mpg") > F.lit(median_mpg), F.lit(1.0)).otherwise(F.lit(0.0))
)

auto.select("mpg", "label").show(10)

# 5) Split train/test (giữ seed để ra kết quả ổn định)
train_auto, test_auto = auto.randomSplit([0.7, 0.3], seed=42)
print("Train =", train_auto.count(), "Test =", test_auto.count())



Median mpg = 22.5
+----+-----+
| mpg|label|
+----+-----+
|18.0|  0.0|
|15.0|  0.0|
|18.0|  0.0|
|16.0|  0.0|
|17.0|  0.0|
|15.0|  0.0|
|14.0|  0.0|
|14.0|  0.0|
|14.0|  0.0|
|15.0|  0.0|
+----+-----+
only showing top 10 rows

Train = 296 Test = 96


**1.2.** Áp dụng Logistic Regression cho tập dữ liệu với các giá trị siêu tham số `regParam` khác nhau để dự đoán `mpg`. Cho biết cross-validation error ứng với các giá trị khác nhau của siêu tham số này. Nhận xét kết quả thu được. Tham khảo document về Logistic Regression của Spark ở [LogisticRegression](https://spark.apache.org/docs/latest/api/python/reference/api/pyspark.ml.classification.LogisticRegression.html#pyspark.ml.classification.LogisticRegression).

In [25]:
# Viết code của bạn ở đây
from pyspark.ml import Pipeline
from pyspark.ml.feature import StringIndexer, OneHotEncoder, VectorAssembler
from pyspark.ml.classification import LogisticRegression
from pyspark.ml.evaluation import MulticlassClassificationEvaluator
from pyspark.ml.tuning import CrossValidator, ParamGridBuilder

# ===== Feature Engineering =====
# origin: 1/2/3 là categorical -> OHE
origin_indexer = StringIndexer(inputCol="origin", outputCol="originIndex", handleInvalid="keep")
origin_ohe = OneHotEncoder(inputCols=["originIndex"], outputCols=["originOHE"], handleInvalid="keep")

feature_cols_num = ["cylinders", "displacement", "horsepower", "weight", "acceleration", "year"]
assembler = VectorAssembler(
    inputCols=feature_cols_num + ["originOHE"],
    outputCol="features",
    handleInvalid="keep"
)

lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=200)

pipeline = Pipeline(stages=[origin_indexer, origin_ohe, assembler, lr])

# ===== Cross Validation on regParam =====
evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

reg_list = [0.0, 0.001, 0.01, 0.05, 0.1, 0.5, 1.0]
paramGrid = (ParamGridBuilder()
    .addGrid(lr.regParam, reg_list)
    .build()
)

cv = CrossValidator(
    estimator=pipeline,
    estimatorParamMaps=paramGrid,
    evaluator=evaluator,
    numFolds=5,
    parallelism=2,
    seed=42
)

cvModel = cv.fit(train_auto)

# Cross-validation results (Spark trả về metric = accuracy trung bình)
avg_metrics = cvModel.avgMetrics  # accuracy
rows = []
for rp, acc in zip(reg_list, avg_metrics):
    cv_error = 1.0 - acc
    rows.append((float(rp), float(acc), float(cv_error)))

result_df = spark.createDataFrame(rows, ["regParam", "cv_accuracy", "cv_error"]).orderBy("regParam")
result_df.show(truncate=False)

# ===== Evaluate best model on test =====
bestModel = cvModel.bestModel
pred_test = bestModel.transform(test_auto)
test_acc = evaluator.evaluate(pred_test)
print("Best Test Accuracy =", test_acc)
print("Best Test Error =", 1.0 - test_acc)


+--------+------------------+-------------------+
|regParam|cv_accuracy       |cv_error           |
+--------+------------------+-------------------+
|0.0     |0.8987098690893758|0.10129013091062422|
|0.001   |0.8924537596074599|0.10754624039254013|
|0.01    |0.8986034922277806|0.10139650777221942|
|0.05    |0.9028524332509154|0.09714756674908465|
|0.1     |0.8999112567803271|0.10008874321967287|
|0.5     |0.896880953750024 |0.10311904624997603|
|1.0     |0.896880953750024 |0.10311904624997603|
+--------+------------------+-------------------+

Best Test Accuracy = 0.9270833333333334
Best Test Error = 0.07291666666666663


## Câu 2 - So sánh các mô hình phân loại

- Thực hiện việc train tất cả các mô hình `LogisticRegression`, `DecisionTreeClassifier` và `RandomForestClassifier`, `GBTClassifier`, `MultilayerPerceptronClassifier`, `LinearSVC`, `NaiveBayes` trên tập dữ liệu HeartDisease (https://archive.ics.uci.edu/ml/datasets/heart+Disease) dùng độ đo Accuracy.

- Điều chỉnh các siêu tham số của các mô hình để chọn mô hình tốt nhất dùng cross validation (tham khảo mục 1.4 ở trên). Để tránh lặp lại các bước xử lý giống nhau nhiều lần như ở trên bạn cần tạo pipeline các bước xử lý. Tham khảo cách tạo pipeline cho mô hình ở https://spark.apache.org/docs/latest/ml-pipeline.html.

- So sánh và nhận xét về kết quả của các mô hình.

- Tham khảo document về các classifier của Spark ở 

    - Classification and Regression ở MLlib Guide: https://spark.apache.org/docs/latest/ml-classification-regression.html

    - Classification module: https://spark.apache.org/docs/latest/api/python/reference/pyspark.ml.html#classification.

Bên dưới là một số gợi ý về khám phá sơ bộ và tiền xử lý dữ liệu.

In [26]:
heart = (spark.read
          .option("HEADER", True)
          .option("inferSchema", True)
          .csv("./data/HeartDisease.csv")
         )

heart.show(5)

+---+---+---+------------+------+----+---+-------+-----+-----+-------+-----+---+----------+---+
|_c0|Age|Sex|   ChestPain|RestBP|Chol|Fbs|RestECG|MaxHR|ExAng|Oldpeak|Slope| Ca|      Thal|AHD|
+---+---+---+------------+------+----+---+-------+-----+-----+-------+-----+---+----------+---+
|  1| 63|  1|     typical|   145| 233|  1|      2|  150|    0|    2.3|    3|  0|     fixed| No|
|  2| 67|  1|asymptomatic|   160| 286|  0|      2|  108|    1|    1.5|    2|  3|    normal|Yes|
|  3| 67|  1|asymptomatic|   120| 229|  0|      2|  129|    1|    2.6|    2|  2|reversable|Yes|
|  4| 37|  1|  nonanginal|   130| 250|  0|      0|  187|    0|    3.5|    3|  0|    normal| No|
|  5| 41|  0|  nontypical|   130| 204|  0|      2|  172|    0|    1.4|    1|  0|    normal| No|
+---+---+---+------------+------+----+---+-------+-----+-----+-------+-----+---+----------+---+
only showing top 5 rows



In [27]:
heart.count()

303

In [28]:
len(heart.columns)

15

In [29]:
heart.dtypes

[('_c0', 'int'),
 ('Age', 'int'),
 ('Sex', 'int'),
 ('ChestPain', 'string'),
 ('RestBP', 'int'),
 ('Chol', 'int'),
 ('Fbs', 'int'),
 ('RestECG', 'int'),
 ('MaxHR', 'int'),
 ('ExAng', 'int'),
 ('Oldpeak', 'double'),
 ('Slope', 'int'),
 ('Ca', 'string'),
 ('Thal', 'string'),
 ('AHD', 'string')]

In [30]:
heart.withColumn('Ca', heart.Ca.cast('int'))

DataFrame[_c0: int, Age: int, Sex: int, ChestPain: string, RestBP: int, Chol: int, Fbs: int, RestECG: int, MaxHR: int, ExAng: int, Oldpeak: double, Slope: int, Ca: int, Thal: string, AHD: string]

In [33]:
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer

heart_raw = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("./data/HeartDisease.csv")
)

# Chuẩn hoá AHD: trim + lowercase để tránh " Yes", "yes", "YES", ...
heart = (heart_raw
    .withColumn("AHD", F.lower(F.trim(F.col("AHD").cast("string"))))
)

# Chỉ giữ 2 lớp hợp lệ (binary) -> đây là điểm quan trọng để LinearSVC chạy được
heart = heart.filter(F.col("AHD").isin(["yes", "no"]))

# (Tuỳ file) xử lý Ca="?" -> null -> drop
heart = (heart
    .withColumn("Ca", F.when(F.col("Ca").cast("string") == "?", F.lit(None)).otherwise(F.col("Ca")))
    .withColumn("Ca", F.col("Ca").cast("int"))
    .dropna(subset=["Ca", "Thal", "ChestPain", "RestECG", "Slope", "AHD"])
)

# Label indexer: TUYỆT ĐỐI không dùng keep cho LABEL (tránh sinh lớp thứ 3)
label_indexer = StringIndexer(inputCol="AHD", outputCol="label", handleInvalid="error")

# Kiểm tra số lớp label (PHẢI = 2)
heart.select("AHD").groupBy("AHD").count().show()


+---+-----+
|AHD|count|
+---+-----+
| no|  161|
|yes|  138|
+---+-----+



In [35]:
# Viết code của bạn ở đây (bạn có thể tạo thêm các cell khác để thực nghiệm và phân tích kết quả)
from pyspark.sql import functions as F
from pyspark.ml.feature import StringIndexer

heart_raw = (spark.read
    .option("header", True)
    .option("inferSchema", True)
    .csv("./data/HeartDisease.csv")
)

# Chuẩn hoá AHD: trim + lowercase để tránh " Yes", "yes", "YES", ...
heart = (heart_raw
    .withColumn("AHD", F.lower(F.trim(F.col("AHD").cast("string"))))
)

# Chỉ giữ 2 lớp hợp lệ (binary) -> đây là điểm quan trọng để LinearSVC chạy được
heart = heart.filter(F.col("AHD").isin(["yes", "no"]))

# (Tuỳ file) xử lý Ca="?" -> null -> drop
heart = (heart
    .withColumn("Ca", F.when(F.col("Ca").cast("string") == "?", F.lit(None)).otherwise(F.col("Ca")))
    .withColumn("Ca", F.col("Ca").cast("int"))
    .dropna(subset=["Ca", "Thal", "ChestPain", "RestECG", "Slope", "AHD"])
)

# Label indexer: TUYỆT ĐỐI không dùng keep cho LABEL (tránh sinh lớp thứ 3)
label_indexer = StringIndexer(inputCol="AHD", outputCol="label", handleInvalid="error")

# Kiểm tra số lớp label (PHẢI = 2)
heart.select("AHD").groupBy("AHD").count().show()


# Categorical -> OHE (theo gợi ý trong đề)
catInputCols = ["ChestPain", "RestECG", "Slope", "Thal"]
catIndexCols = [c + "Index" for c in catInputCols]
catOheCols   = [c + "OHE" for c in catInputCols]

indexers = [StringIndexer(inputCol=c, outputCol=c+"Index", handleInvalid="keep") for c in catInputCols]
ohe = OneHotEncoder(inputCols=catIndexCols, outputCols=catOheCols, handleInvalid="keep")

# Numeric columns (bỏ _c0 nếu có)
ignore_cols = set(["_c0", "AHD"])
all_cols = [c for c in heart.columns if c not in ignore_cols]
num_cols = [c for c, t in heart.dtypes if (c in all_cols and t in ("int", "double")) and c != "label"]

# Assemble features
assembler = VectorAssembler(
    inputCols=num_cols + catOheCols,
    outputCol="features",
    handleInvalid="keep"
)

# Split train/test
train_heart, test_heart = heart.randomSplit([0.7, 0.3], seed=42)

evaluator = MulticlassClassificationEvaluator(labelCol="label", predictionCol="prediction", metricName="accuracy")

# =========================
# 2) Helper: Train CV
# =========================
def train_with_cv(model_name, estimator, paramGrid, seed=42):
    """
    Returns: (model_name, best_cv_acc, test_acc, best_params_dict)
    """
    pipe = Pipeline(stages=[label_indexer] + indexers + [ohe, assembler, estimator])

    cv = CrossValidator(
        estimator=pipe,
        estimatorParamMaps=paramGrid,
        evaluator=evaluator,
        numFolds=5,
        parallelism=2,
        seed=seed
    )

    cvModel = cv.fit(train_heart)

    bestModel = cvModel.bestModel
    pred_test = bestModel.transform(test_heart)
    test_acc = evaluator.evaluate(pred_test)

    # best params
    # last stage is the fitted estimator model
    fitted_est = bestModel.stages[-1]
    best_params = {}
    for p in paramGrid[0].keys():
        # CrossValidator doesn't directly expose best param map nicely,
        # but we can read from fitted estimator using getOrDefault
        best_params[p.name] = fitted_est.getOrDefault(p)

    best_cv_acc = float(max(cvModel.avgMetrics))
    return (model_name, best_cv_acc, float(test_acc), best_params)

# =========================
# 3) Define models + grids
# =========================
results = []

# (1) Logistic Regression
lr = LogisticRegression(featuresCol="features", labelCol="label", maxIter=200)
lr_grid = (ParamGridBuilder()
    .addGrid(lr.regParam, [0.0, 0.01, 0.1])
    .addGrid(lr.elasticNetParam, [0.0, 0.5, 1.0])
    .build()
)
results.append(train_with_cv("LogisticRegression", lr, lr_grid))

# (2) Decision Tree
dt = DecisionTreeClassifier(featuresCol="features", labelCol="label")
dt_grid = (ParamGridBuilder()
    .addGrid(dt.maxDepth, [3, 5, 8])
    .addGrid(dt.minInstancesPerNode, [1, 5, 10])
    .build()
)
results.append(train_with_cv("DecisionTree", dt, dt_grid))

# (3) Random Forest
rf = RandomForestClassifier(featuresCol="features", labelCol="label", seed=42)
rf_grid = (ParamGridBuilder()
    .addGrid(rf.numTrees, [50, 100])
    .addGrid(rf.maxDepth, [5, 8])
    .build()
)
results.append(train_with_cv("RandomForest", rf, rf_grid))

# (4) GBT (binary classification OK)
gbt = GBTClassifier(featuresCol="features", labelCol="label", seed=42)
gbt_grid = (ParamGridBuilder()
    .addGrid(gbt.maxIter, [30, 60])
    .addGrid(gbt.maxDepth, [3, 5])
    .build()
)
results.append(train_with_cv("GBTClassifier", gbt, gbt_grid))

# (5) MLP
# input size = num_features sau khi OHE => cần fit pipeline 1 lần để biết vector size
tmp_pipe = Pipeline(stages=[label_indexer] + indexers + [ohe, assembler])
tmp_model = tmp_pipe.fit(train_heart)
tmp_df = tmp_model.transform(train_heart).select("features").limit(1).collect()
input_dim = int(tmp_df[0]["features"].size)

# Chọn cấu hình layers hợp lý
layers = [input_dim, 16, 8, 2]
mlp = MultilayerPerceptronClassifier(featuresCol="features", labelCol="label", seed=42)
mlp_grid = (ParamGridBuilder()
    .addGrid(mlp.layers, [layers])
    .addGrid(mlp.maxIter, [100, 200])
    .addGrid(mlp.stepSize, [0.03, 0.1])
    .build()
)
results.append(train_with_cv("MultilayerPerceptron", mlp, mlp_grid))

# (6) LinearSVC (binary)
svc = LinearSVC(featuresCol="features", labelCol="label", maxIter=200)
svc_grid = (ParamGridBuilder()
    .addGrid(svc.regParam, [0.01, 0.1, 1.0])
    .build()
)
results.append(train_with_cv("LinearSVC", svc, svc_grid))

# (7) Naive Bayes (yêu cầu features không âm -> ta KHÔNG scale)
nb = NaiveBayes(featuresCol="features", labelCol="label", modelType="multinomial")
nb_grid = (ParamGridBuilder()
    .addGrid(nb.smoothing, [0.5, 1.0, 2.0])
    .build()
)
results.append(train_with_cv("NaiveBayes", nb, nb_grid))

# =========================
# 4) Show comparison table
# =========================
rows = [(name, float(cv_acc), float(test_acc), str(params)) for (name, cv_acc, test_acc, params) in results]
summary = spark.createDataFrame(rows, ["Model", "Best_CV_Accuracy", "Test_Accuracy", "Best_Params"]) \
              .orderBy(F.desc("Test_Accuracy"))

summary.show(truncate=False)

best = summary.limit(1).collect()[0]
print("BEST MODEL =", best["Model"], "| Test_Accuracy =", best["Test_Accuracy"])


+---+-----+
|AHD|count|
+---+-----+
| no|  161|
|yes|  138|
+---+-----+

+--------------------+------------------+------------------+------------------------------------------------------------+
|Model               |Best_CV_Accuracy  |Test_Accuracy     |Best_Params                                                 |
+--------------------+------------------+------------------+------------------------------------------------------------+
|MultilayerPerceptron|0.6105612424445157|0.8717948717948718|{'layers': [29, 16, 8, 2], 'maxIter': 200, 'stepSize': 0.03}|
|LinearSVC           |0.8359609467348836|0.8461538461538461|{'regParam': 1.0}                                           |
|LogisticRegression  |0.8347243545370289|0.8461538461538461|{'regParam': 0.1, 'elasticNetParam': 0.5}                   |
|RandomForest        |0.8263369774969351|0.8461538461538461|{'numTrees': 100, 'maxDepth': 5}                            |
|GBTClassifier       |0.7843518815717727|0.8205128205128205|{'maxIter': 3